# Multi-Turn Conversations — Claude Managed Agents

A session is **stateful**. After the agent reaches `status_idle`, you can send another message and it will remember everything said so far — Anthropic maintains the full conversation history server-side.

This notebook shows:
1. How to send multiple messages to the same session
2. That context carries over automatically — no need to resend history
3. A reusable `chat()` helper to reduce boilerplate
4. An optional interactive REPL you can run in JupyterLab

**Session lifecycle reminder:**

| Event | Meaning |
|---|---|
| `session.status_idle` | Agent finished its turn — **you can send another message** |
| `session.status_terminated` | Session is closed — no more messages possible |

## 1. Setup

Same as module 01: install deps and load the API key from the project-root `.env`.

In [1]:
%pip install -q -U anthropic python-dotenv --user


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()
print("Client ready")

Client ready


## 2. Create Environment, Agent, and Session

Same setup as module 01. The key here is creating **one session** that we'll reuse for all our turns.

> **Production note:** `environment.id` and `agent.id` should be persisted (e.g. in a database or config file) so you don't recreate them on every run. A session is single-use — create a new one per conversation.

In [3]:
environment = client.beta.environments.create(
    name="multi-turn-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

agent = client.beta.agents.create(
    name="Multi-Turn Agent",
    model="claude-opus-4-7",
    system="You are a helpful assistant. Keep answers concise.",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": True},
        }
    ],
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
)

print(f"Environment : {environment.id}")
print(f"Agent       : {agent.id}  (v{agent.version})")
print(f"Session     : {session.id}  [{session.status}]")

Environment : env_01FQuinxmhw66fuHao1z3Hsh
Agent       : agent_013pDTK1V8V3RhGsERmzMV4E  (v1)
Session     : sesn_01GkeSNtXo6krUKGFwGDP3df  [idle]


## 3. A Reusable `chat()` Helper

The stream-first pattern from module 01 is the same every time — open stream, send message, iterate events. Let's wrap it in a helper so we can focus on the conversation instead of the plumbing.

The helper:
- Always follows the stream-first order
- Streams text to stdout as it arrives
- Returns the full response as a string
- Stops on both `status_idle` and `status_terminated`

In [4]:
def chat(text: str) -> str:
    """Send a message and stream the response. Returns full response text."""
    parts = []

    with client.beta.sessions.events.stream(session.id) as stream:
        client.beta.sessions.events.send(
            session_id=session.id,
            events=[
                {
                    "type": "user.message",
                    "content": [{"type": "text", "text": text}],
                }
            ],
        )

        print(f"You: {text}")
        print("Agent: ", end="", flush=True)

        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        parts.append(block.text)
                        print(block.text, end="", flush=True)

            elif event.type in ("session.status_idle", "session.status_terminated"):
                break

    print("\n")
    return "".join(parts)


print("chat() helper ready")

chat() helper ready


## 4. Turn 1 — Start the Conversation

Ask a simple question to kick things off.

In [5]:
response1 = chat("What's the largest planet in our solar system?")

You: What's the largest planet in our solar system?
Agent: Jupiter.



## 5. Turn 2 — Context Is Preserved

The follow-up question deliberately omits any mention of the planet. The agent can only answer correctly if it remembers the previous exchange — which it does, because the session holds the full history.

Notice: you send only the new message. You never resend previous turns.

In [6]:
response2 = chat("How many moons does it have?")

You: How many moons does it have?
Agent: Jupiter has 95 confirmed moons (as recognized by the IAU).



## 6. Turn 3 — Build Deeper

Another follow-up that only makes sense in context. "The three largest ones" refers back to the moons mentioned in turn 2.

In [7]:
response3 = chat("Name the three largest ones and one interesting fact about each.")

You: Name the three largest ones and one interesting fact about each.
Agent: The three largest moons of Jupiter (all part of the four Galilean moons) are:

1. **Ganymede** – The largest moon in the entire solar system, even bigger than the planet Mercury. It's also the only moon known to generate its own magnetic field.

2. **Callisto** – One of the most heavily cratered objects in the solar system, with a surface that's remained largely unchanged for about 4 billion years.

3. **Io** – The most volcanically active body in the solar system, with hundreds of active volcanoes driven by tidal heating from Jupiter's immense gravity.



## 7. Optional: Interactive REPL

Run this cell in JupyterLab for a live chat experience. Type `quit` (or `exit`) to stop.

> **Note:** `input()` works in JupyterLab and VS Code notebooks. It may not work in some headless or CI environments.

In [8]:
# Create a fresh session for the REPL so it starts with a clean slate
repl_session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
)

print(f"REPL session: {repl_session.id}")
print("Type 'quit' or 'exit' to stop.\n")

terminated = False

while not terminated:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    print("Agent: ", end="", flush=True)

    with client.beta.sessions.events.stream(repl_session.id) as stream:
        client.beta.sessions.events.send(
            session_id=repl_session.id,
            events=[
                {
                    "type": "user.message",
                    "content": [{"type": "text", "text": user_input}],
                }
            ],
        )

        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        print(block.text, end="", flush=True)

            elif event.type == "session.status_idle":
                break

            elif event.type == "session.status_terminated":
                terminated = True
                break

    print("\n")

REPL session: sesn_017x8ayGMFdJx1fv9rBpRy9f
Type 'quit' or 'exit' to stop.



KeyboardInterrupt: Interrupted by user

## Summary

Multi-turn works by reusing the same session ID across multiple send/stream cycles:

```
session = sessions.create(...)          # one session for the whole conversation

# Turn 1
with events.stream(session.id) as s:
    events.send(session_id, message_1)
    for event in s: ...                 # wait for status_idle

# Turn 2 — same session, context preserved automatically
with events.stream(session.id) as s:
    events.send(session_id, message_2)
    for event in s: ...                 # wait for status_idle
```

Key takeaways:
- `status_idle` means "ready for another message" — the session is still alive
- `status_terminated` means the session is closed — create a new one
- You never resend previous messages — Anthropic maintains history server-side
- Wrap the plumbing in a helper to keep conversation code clean

### Next steps
- **03-tools** — give the agent bash, file, and code-execution tools inside the container
- **04-mcp** — connect external MCP servers to extend what the agent can do